# Population-scale noter extraction

Run the deterministic Tesseract pipeline + noter-parser end-to-end across many orgnrs in a single Colab session. Designed for **Colab Pro** (8 cores) but scales linearly with available CPU.

Pipeline:
1. Download regnskap PDF for each (orgnr, year)
2. Tesseract Norwegian OCR with TSV column-aware reconstruction
3. Note segmenter → kv_extractor → write tesseract_v1 JSON to GCS
4. noter-parser parses to flat CSV tables
5. Pandera-style validators flag arithmetic-failed rows

Cost: $0 (free tier OK) — pure CPU compute, no LLM calls.

In [ ]:
# 1. System dependencies
!apt-get install -qq tesseract-ocr-nor poppler-utils
!pip install -q rapidfuzz google-cloud-storage pdfplumber pypdfium2

In [ ]:
# 2. Optional: enable Tier 2 semantic fallback (BGE-M3)
#    Skip if you only need the deterministic + fuzzy layers.
!pip install -q sentence-transformers torch

In [ ]:
# 3. Clone both repos
!git clone https://github.com/sondreskarsten/noter-text-extraction.git
!git clone https://github.com/sondreskarsten/noter-parser.git
!pip install -e ./noter-text-extraction -q
!pip install -e ./noter-parser -q

In [ ]:
# 4. Auth
from google.colab import auth
auth.authenticate_user()
import os
os.environ['GOOGLE_CLOUD_PROJECT'] = 'sondreskarsten-d7d14'

In [ ]:
# 5. Pre-warm the embedding index (one-time, ~30s)
from noter_parser import schema_mapper
schema_mapper._build_embedded_index()
print('Embedding index ready.')

In [ ]:
# 6. Define the work list
ORGNR_YEAR_PAIRS = [
    ('989100106', 2024),
    ('989100106', 2023),
    # ... add more
]

# Or load from a file:
# with open('orgnrs.txt') as f:
#     ORGNR_YEAR_PAIRS = [tuple(ln.strip().split()) for ln in f if ln.strip()]

In [ ]:
# 7. Run text extraction in parallel
from concurrent.futures import ThreadPoolExecutor, as_completed
from noter_text_extraction import extract_one

def _ext(spec):
    orgnr, year = spec
    try:
        payload = extract_one(orgnr, int(year), upload=True)
        return (orgnr, year, 'ok', payload['n_notes'],
                sum(len(n['raw_amounts']) for n in payload['noter']))
    except Exception as e:
        return (orgnr, year, f'err: {type(e).__name__}: {e}', 0, 0)

results = []
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = {ex.submit(_ext, s): s for s in ORGNR_YEAR_PAIRS}
    for fut in as_completed(futures):
        r = fut.result()
        print(f'  {r[0]} {r[1]}  {r[2]}  notes={r[3]}  amts={r[4]}')
        results.append(r)

ok = sum(1 for r in results if r[2] == 'ok')
print(f'\nOCR phase: {ok}/{len(results)} succeeded')

In [ ]:
# 8. Parse all extracted JSONs through noter-parser
from noter_parser import parse_orgnr_with_validation
from noter_parser.sources import load_noter_json
from collections import defaultdict
import pandas as pd

def loader(orgnr, year):
    return load_noter_json(orgnr, year)

# Group years per orgnr
by_orgnr = defaultdict(list)
for orgnr, year, *_ in results:
    by_orgnr[orgnr].append(int(year))

all_tables = defaultdict(list)
all_validation = []
for orgnr, years in by_orgnr.items():
    res = parse_orgnr_with_validation(loader, orgnr, years)
    for table, rows in res['tables'].items():
        all_tables[table].extend(rows)
    for table, rep in res['validation'].items():
        all_validation.append({
            'orgnr': orgnr, 'table': table,
            **{k: v for k, v in rep.items() if k != 'rows'},
        })

# Print summary
for table, rows in all_tables.items():
    print(f'  {table}: {len(rows)} rows')

# Validation summary
val_df = pd.DataFrame(all_validation)
print('\nValidation pass rates:')
print(val_df.groupby('table')[['n_passed', 'n_failed', 'pass_rate']].mean())

In [ ]:
# 9. Write per-table CSVs to GCS
from google.cloud import storage
import io

client = storage.Client()
bkt = client.bucket('sondre_brreg_data')

for table, rows in all_tables.items():
    if not rows:
        continue
    # Group by orgnr — one CSV per (table, orgnr)
    by_or = defaultdict(list)
    for r in rows:
        by_or[r['orgnr']].append(r)
    for orgnr, orows in by_or.items():
        df = pd.DataFrame(orows)
        buf = io.StringIO()
        df.to_csv(buf, index=False)
        target = f'raw/noter_extraction_2025/structured/{table}/{orgnr}.csv'
        bkt.blob(target).upload_from_string(buf.getvalue(), content_type='text/csv')
    print(f'  {table}: wrote {len(by_or)} CSV files')

In [ ]:
# 10. Write review.csv for failed validations (each row = one failed identity check)
review = []
for orgnr, years in by_orgnr.items():
    res = parse_orgnr_with_validation(loader, orgnr, years)
    for table, rep in res['validation'].items():
        for r in rep['rows']:
            for ident in r['identities']:
                if not ident['passed']:
                    review.append({
                        'orgnr': orgnr, 'table': table,
                        'identity': ident['name'],
                        'expected': ident['expected'],
                        'actual': ident['actual'],
                        'diff': ident['diff'],
                        'row_year': r['row'].get('report_year'),
                    })
rev_df = pd.DataFrame(review)
buf = io.StringIO()
rev_df.to_csv(buf, index=False)
bkt.blob('raw/noter_extraction_2025/review/identity_failures.csv') \
    .upload_from_string(buf.getvalue(), content_type='text/csv')
print(f'Wrote {len(rev_df)} identity failures to review queue')